In [5]:
# Import libraries
import requests
from bs4 import BeautifulSoup
import dateutil.parser as dparser

# URL from which pdfs to be downloaded
url = "https://official.nba.com/nba-injury-report-2024-25-season/"

# Requests URL and get response object
response = requests.get(url)

# Parse text obtained
soup = BeautifulSoup(response.text, 'html.parser')

# Find all hyperlinks present on webpage
links = soup.find_all('a')

readings_time = {}; readings_pdf = {}
for link in links:
    if link.decode_contents().endswith('ET report'):
        readings_time[link.contents[0]] = dparser.parse(link.contents[0], fuzzy=True, ignoretz=True)
        readings_pdf[link.contents[0]] = requests.get(link.get('href'))

pdf = open('injury.pdf', 'wb')
pdf.write(readings_pdf[max(readings_time, key = readings_time.get)].content)
pdf.close()

In [312]:
import pandas as pd
import numpy as np
import tabula
from dataHub import dataHub
from sqlalchemy.dialects.postgresql.base import PGDialect
PGDialect._get_server_version_info = lambda *args: (9, 2)
dh = dataHub()

db_con = dh.db_connect('cockroach')

top = 75
left = 19 
width = 804
height = 438

dfs = tabula.read_pdf('injury.pdf', area=[top, left, top+height, left+width], pages = "all")

status_true = pd.DataFrame({'status': ['Available', 'Probable', 'Questionable', 'Out']})
teams_true = pd.read_sql("SELECT CONCAT(team_long, ' ', team_name) AS teams FROM nba.teams", db_con)

In [314]:
for df in dfs:
    
    if(len(df.columns) < len(dfs[0].columns)):

        # Move column to row
        colnames_temp = ['Unnamed: ' + str(el) for el in list(range(1, len(df.columns)+1))]
        df = (pd.concat([
            df.set_axis(colnames_temp, axis=1),
            pd.DataFrame({key : np.nan if 'Unnamed' in val else val for key, val in dict(zip(colnames_temp, df.columns)).items()}, index=[0])
        ]))
        
        # test for date
        if (False in ['/' in el for el in df.iloc[:, 0] if el is not np.nan]):
            df.insert(loc=0, column='Game Date', value=np.nan)
            # cols_insert_obj['Game Date'] = 1
        else:
            df.columns.values[0] = 'Game Date'
            
        # test for time
        if (False in [':' in el for el in df.iloc[:, 1] if el is not np.nan]):
            df.insert(loc=1, column='Game Time', value=np.nan)
        else:
            df.columns.values[1] = 'Game Time'

        # test for matchup
        if (False in ['@' in el for el in df.iloc[:, 2] if el is not np.nan]):
            df.insert(loc=2, column='Matchup', value=np.nan)
        else:
            df.columns.values[2] = 'Matchup'
            
        # test for team
        teams_temp = pd.DataFrame({'teams': [el for el in dfs[0].iloc[:, 3] if el is not np.nan]}).value_counts().reset_index()
        if (pd.merge(teams_true, teams_temp, on='teams', how='left')['count'].sum()):
           df.insert(loc=3, column='Team', value=np.nan)
        else:
            df.columns.values[3] = 'Team'
            
        # test for player name
        if (False in [',' in el for el in df.iloc[:, 4] if el is not np.nan]):
            df.insert(loc=4, column='Player Name', value=np.nan)
        else:
            df.columns.values[4] = 'Player Name'
            
        # test for status
        status_temp = pd.DataFrame({'status': [el for el in dfs[0].iloc[:, 5].to_list() if el is not np.nan]}).value_counts().reset_index()
        if (pd.merge(status_true, status_temp, on='status', how='left')['count'].sum() == 0):
            df.insert(loc=5, column='Current Status', value=np.nan)
        else:
            df.columns.values[5] = 'Current Status'

        # test for injury
        df.columns.values[6] = 'Current Status'

    print(df)

     Game Date   Game Time  Matchup                   Team  \
0   03/15/2025  06:00 (ET)  BOS@BKN         Boston Celtics   
1          NaN         NaN      NaN          Brooklyn Nets   
2          NaN         NaN      NaN                    NaN   
3          NaN         NaN      NaN                    NaN   
4          NaN         NaN      NaN                    NaN   
5          NaN         NaN      NaN                    NaN   
6          NaN         NaN      NaN                    NaN   
7          NaN         NaN      NaN                    NaN   
8          NaN         NaN      NaN                    NaN   
9          NaN         NaN      NaN                    NaN   
10         NaN         NaN      NaN                    NaN   
11         NaN         NaN      NaN                    NaN   
12         NaN         NaN      NaN                    NaN   
13         NaN  07:00 (ET)  OKC@DET  Oklahoma City Thunder   
14         NaN         NaN      NaN                    NaN   
15      